# DSPL cosmology sampling — ported to the new (scene) gigalens API

This notebook ports `GIGALens-Code-paper/experiments/sample_cosmology/dspl(1).ipynb`
(old API, on the `paper-oldapi` world) to the new **scene** API
(`gigalens.jax.scene` / `gigalens.jax.scene_prob_model`), so the system can be run
through the sampling-diagnostic suite in `gigalens_research` (`Pipeline` /
`PosteriorReport` / `PipelineReport`).

**System, unchanged from the old notebook:** a single EPL deflector at
`z_lens = 0.5`, and **two independent Sersic sources** at `z_source1 = 1.0` and
`z_source2 = 1.5`, each rendered as its own noisy "band" (its own image, its own
Poisson+background noise) through the *same* shared lens. Free parameters: the
lens mass profile, both sources' light profiles, and **`Om0`, `w0`** of a flat
`w0waCDM` cosmology (`H0`, `k`, `wa` fixed, exactly as in the old notebook). The
cosmology enters only through each source plane's **deflection ratio**
(`D_LS/D_S`, normalized to `z_source1`) — this is the classical
"double source-plane" cosmography test (e.g. Collett & Auger 2014), not full
recursive multi-plane ray-tracing (neither source plane itself carries mass).

**Porting verdict (read before trusting anything below):** the new API's scene
module supports this system *natively* — `LensModel([Plane(mass=...),
Plane(light=...), Plane(light=...)], cosmo=Component(...))` with a **single**
mass plane and a cosmology present automatically selects the
`deflection_ratio` trace mode, and the per-plane scaling is computed from
`cosmo.profile.deflection_ratio(z_source, **cosmo_params)` — the *same*
`gigalens/cosmo.py` formula the old notebook calls directly. There is
**no capability gap** to route around here; no adapter was needed. See the
"API mapping" and "Caveats" markdown cells throughout for the handful of
places where old→new behavior differs numerically or structurally.

**This notebook was NOT executed** (no MAP optimization or MCLMC sampling was
run) — login-node policy. Construction (model, prob model, pipeline wiring)
and a couple of single/cheap forward-simulate calls were smoke-tested
separately in the sibling `gigalens_multinode_env` (Shifter+JAX 0.10) session
that produced this port; see the porting report for details.


In [ ]:
import jax
jax.config.update("jax_enable_x64", True)


In [ ]:
import os
import time
from datetime import datetime

import numpy as np
import jax.numpy as jnp
import optax
import matplotlib.pyplot as plt

import tensorflow_probability.substrates.jax as tfp
tfd = tfp.distributions
tfb = tfp.bijectors

# --- new (scene) API ---------------------------------------------------------
from gigalens.jax.scene import Component, Plane, LensModel
from gigalens.jax.profiles.mass.epl import EPL
from gigalens.jax.profiles.light.sersic import SersicEllipse
from gigalens.jax.cosmo import w0waCDM_Cosmo
from gigalens.jax.scene_prob_model import ImageData, ProbModel
from gigalens.jax.scene_simulator import SceneSimulator
from gigalens.simulator import SimulatorConfig
from gigalens.jax.inference import ModellingSequence

# --- research-side pipeline / diagnostics (gigalens_research) ----------------
from gigalens_research.inference_utils import (
    InferenceContext, Pipeline, MAPStage, BridgeStage, MCLMCStage,
)
from gigalens_research.plotting import PosteriorReport, PipelineReport


## API mapping: old → new (summary)

| Old API (`gigalens-old` / `paper-oldapi`) | New API (`gigalens` scene module, this notebook) |
|---|---|
| `gigalens.jax.prior.Prior(profile, dict(...))` | `gigalens.jax.scene.Component(profile, dict(...))` |
| `Prior(profile, dict(z_source=Z, **light_priors))` for a source | `Plane(redshift=Z, light=[Component(profile, light_priors)])` — geometry moves off the profile/prior and onto the `Plane` |
| `make_prior_and_model(lenses=[...], sources=[...], cosmo=cosmo_prior)` | `LensModel([Plane(redshift=z_lens, mass=[...]), Plane(redshift=z_s1, light=[...]), Plane(redshift=z_s2, light=[...])], cosmo=Component(cosmo_profile, dict(...)))` |
| `gigalens.jax.cosmo.w0waCDM_Cosmo` (and `wCDM_Cosmo`) | **Unchanged** — same class, same file layout (`gigalens/jax/cosmo.py`), same `efunc`/`deflection_ratio` math in both worlds |
| `cosmo_model.deflection_ratio(z_source, **cosmo_params)` called by hand in a plot cell | Called automatically, once per lensed light `Plane`, inside `SceneSimulator._trace_deflection_ratio` — same underlying method |
| `mass.epl.EPL()`, `light.sersic.SersicEllipse(use_lstsq=False)` | Unchanged profile classes/params (`EPL._params`, `SersicEllipse._params` + auto-appended `Ie` when `use_lstsq=False`) |
| `LensSimulator(phys_model, sim_config, bs=1)` | `SceneSimulator(model, sim_config, sees=[...])` — one simulator **per dataset/band**, each restricted to the light it "sees" (`sees`) but tracing through the full shared mass |
| `ForwardMultiModel(prior, observed_images, background_rmss=[...], exp_time=..., include_pixels=True, include_positions=False)` | `ProbModel(model, datasets=[Dataset(img, sim_config, background_rms=..., exp_time=..., sees=[...]), ...], mode="forward")` — one `Dataset` per band; `mode="forward"` = amplitudes sampled (old `ForwardProbModel`), `mode="lstsq"` = amplitudes solved (old `BackwardProbModel`). There is no separate `include_positions` knob (only the pixel likelihood is modeled) — irrelevant here since the old notebook already had `include_positions=False`. |
| `ModellingSequence(phys_model, prob_model, sim_config)` | `ModellingSequence(prob_model)` — scene-only; `phys_model`/`sim_config` are derived from `prob_model` |
| `model_seq.MAP(...)` / `model_seq.SVI(...)` (called by hand) | Wrapped as pipeline `Stage`s (`MAPStage`, `SVIStage`, `HessianSurrogateStage`, `BridgeStage`) run through `Pipeline(ctx).add(...).run(...)`, which also writes a **model card** and caches each stage to disk |
| `gigalens_research.inference.MCLMC_JIT` called by hand | `MCLMCStage(...)` — thin wrapper around the same `MCLMC_JIT` |
| Hand-rolled `corner()` / `az.rhat()` / `az.ess()` cells | `PipelineReport(pipeline).diagnostics("mclmc", ...)` and `PosteriorReport(pipeline.posterior())` from `gigalens_research.plotting`, matching the diagnostic-suite idiom used in `experiments/sim_carousel/carousel_sampling_minimal_example.ipynb` and `experiments/TestNewAPI/TestNewAPI.ipynb` |

**Not carried over:** the old notebook's own **SVI** stage (it built a
`qz` surrogate via `model_seq.SVI(...)` before MCLMC). Per the user's stated
preference for this repo (MAP → MCLMC, small fixed-covariance `qz`, no SVI),
this port instead seeds MCLMC from a small diagonal Gaussian around the MAP
optimum (`BridgeStage`), the same idiom used in
`carousel_sampling_minimal_example.ipynb`. This is a genuine behavioral
difference from the old notebook, called out again at the MCLMC cell below.


## Physical model — lens, two sources, cosmology

Priors, parameter names, and hardcoded values are carried over **verbatim**
from the old notebook (`Prior(...)` → `Component(...)`, same `tfd`
distributions, same numbers). This includes the old notebook's custom
`UniformBij` (a `tfd.Uniform` with a `Shift→Scale→NormalCDF` event-space
bijector instead of TFP's default sigmoid-based one for `Uniform`) for `Om0`
and `w0` — it is passed straight through, since `LensModel`/`Component` accept
any `tfd.Distribution` instance.

The old notebook's lens `shear` component (`mass.shear.Shear()`) was commented
out of the model (`lenses=[lens#, shear]`) — so it is **omitted here too**,
matching what was actually run, not what was drafted.


In [ ]:
z_lens = 0.5
z_source1 = z_lens * 2
z_source2 = z_lens * 3

lens = Component(
    EPL(),
    dict(
        theta_E=tfd.LogNormal(jnp.log(1.25), 0.25),
        gamma=tfd.TruncatedNormal(2, 0.25, 1, 3),
        e1=tfd.TruncatedNormal(0, 0.1, -0.3, 0.3),
        e2=tfd.TruncatedNormal(0, 0.1, -0.3, 0.3),
        center_x=tfd.Normal(0, 0.05),
        center_y=tfd.Normal(0, 0.05),
    ),
)
# NOTE: the old notebook also drafted a `shear` Prior/Component (mass.shear.Shear())
# but commented it out of the model it actually built and ran; omitted here too.

source1 = Component(
    SersicEllipse(use_lstsq=False),
    dict(
        center_x=tfd.Normal(0, 2),
        center_y=tfd.Normal(0, 2),
        e1=tfd.TruncatedNormal(0., 0.1, -0.3, 0.3),
        e2=tfd.TruncatedNormal(0., 0.1, -0.3, 0.3),
        n_sersic=tfd.Uniform(1, 10),
        R_sersic=tfd.LogNormal(jnp.log(1.), 0.15),
        Ie=tfd.LogNormal(jnp.log(150), 1),
    ),
)
source2 = Component(
    SersicEllipse(use_lstsq=False),
    dict(
        center_x=tfd.Normal(0, 2),
        center_y=tfd.Normal(0, 2),
        e1=tfd.TruncatedNormal(0., 0.1, -0.3, 0.3),
        e2=tfd.TruncatedNormal(0., 0.1, -0.3, 0.3),
        n_sersic=tfd.Uniform(1, 10),
        R_sersic=tfd.LogNormal(jnp.log(1.), 0.15),
        Ie=tfd.LogNormal(jnp.log(150), 1),
    ),
)


def tNCDF_bij(low, high):
    return tfb.Chain([tfb.Shift(low), tfb.Scale(high - low), tfb.NormalCDF()])


# Verbatim from the old notebook: a tfd.Uniform with a Shift/Scale/NormalCDF
# event-space bijector in place of TFP's default (sigmoid-based) one for Uniform.
class UniformBij(tfd.Uniform):
    def __init__(self, *args, event_space_bijector_class=tNCDF_bij, **kwargs):
        self._esb = event_space_bijector_class(*args)
        super().__init__(*args, **kwargs)

    def _default_event_space_bijector(self):
        return self._esb

from gigalens_research.priors import ratio_coords
u_fn = ratio_coords.deflection_ratio_u_fn(w0waCDM_Cosmo(z_lens, z_source1), [z_source2], [1.0], fixed=dict(H0=70.0, k=0.0, wa=0.0))
cosmo = Component(
    w0waCDM_Cosmo(z_lens=z_lens, z_source_ref=z_source1),
    {
        "H0":70.,
        # ("Om0", "w0"): ratio_coords.UFirstRatioCoordsUniform(u_fn, (0.01, 0.99), (-2.0, -1/3), du_dw_atol=1.8e-3, excursion_atol=2e-4, curve_atol=0.0),
        "Om0":UniformBij(jnp.float64(0.1), jnp.float64(0.8)),
        "w0":UniformBij(jnp.float64(-2.0), jnp.float64(-1 / 3)),
        "wa":0.0,
        "k":0.0,
    },
)


# "wa": 

model = LensModel(
    [
        Plane(redshift=z_lens, mass=[lens]),
        Plane(redshift=z_source1, light=[source1]),
        Plane(redshift=z_source2, light=[source2]),
    ],
    cosmo=cosmo,
)

print(f"num_free_params = {model.num_free_params}")
print("z_param_names   =", model.z_param_names)


In [ ]:
truth_scene = {
    "planes": {
        0: {
            "geometry": {"redshift": z_lens},
            "mass": {
                0: {"theta_E": 1.1, "gamma": 2.0, "e1": 0.05, "e2": 0.02,
                    "center_x": 0.0, "center_y": 0.0},
            },
        },
        1: {
            "geometry": {"redshift": z_source1},
            "light": {
                0: {"R_sersic": 0.25, "n_sersic": 2., "e1": 0.05, "e2": 0.,
                    "center_x": 0.05, "center_y": 0., "Ie": 50.},
            },
        },
        2: {
            "geometry": {"redshift": z_source2},
            "light": {
                0: {"R_sersic": 1., "n_sersic": 6., "e1": 0.0, "e2": 0.05,
                    "center_x": 0., "center_y": 0.05, "Ie": 15.},
            },
        },
    },
    "cosmo": dict(H0=70.0, Om0=0.3, k=0.0, w0=-1.0, wa=0.0),
}


In [ ]:
numPix, deltaPix, exp_time, background_rms = 60, 0.065, 1000, 0.1
extent = (-numPix / 2 * deltaPix, numPix / 2 * deltaPix,
          -numPix / 2 * deltaPix, numPix / 2 * deltaPix)

import photutils.psf as psf
kernel = psf.GaussianPSF(x_fwhm=2, y_fwhm=2)
yy, xx = np.mgrid[-7:8, -7:8]
kernel = kernel(xx, yy)

sim_config = SimulatorConfig(
    delta_pix=deltaPix,
    num_pix=numPix,
    kernel=kernel,
    supersample=1,
    likelihood_precision="float64",
)

sim_config_truth = SimulatorConfig(
    delta_pix=deltaPix,
    num_pix=numPix,
    kernel=kernel,
    supersample=1,
    likelihood_precision="float64",
)


In [ ]:
from lenstronomy.Util import image_util


def add_noise(img, exp_time, sigma_bkd):
    poisson = image_util.add_poisson(img, exp_time=exp_time)
    bkg = image_util.add_background(img, sigma_bkd=sigma_bkd)
    return img + poisson + bkg


sim1 = SceneSimulator(model, sim_config_truth, sees=[source1])
sim2 = SceneSimulator(model, sim_config_truth, sees=[source2])
print("trace mode (both simulators, same single-mass-plane model):",
      sim1.trace_mode, sim2.trace_mode)

img1 = np.asarray(sim1.simulate(truth_scene))
img2 = np.asarray(sim2.simulate(truth_scene))

observed_image1 = add_noise(img1, exp_time=exp_time, sigma_bkd=background_rms)
observed_image2 = add_noise(img2, exp_time=exp_time, sigma_bkd=background_rms)

plt.figure(figsize=(8, 3))
ax = plt.subplot(121)
plt.imshow(observed_image1, extent=extent)
plt.colorbar()
ax = plt.subplot(122)
plt.imshow(observed_image2, extent=extent)
plt.colorbar()
plt.show()


## Probabilistic model

`mode="forward"` matches the old notebook's `ForwardMultiModel` (light
amplitudes, `Ie`, are sampled rather than lstsq-solved — consistent with
`SersicEllipse(use_lstsq=False)` above). `Dataset(..., background_rms=...,
exp_time=...)` computes `error_map = sqrt(background_rms**2 +
clip(image, 0, inf) / exp_time)` — checked against
`gigalens-old/src/gigalens/jax/prob_model.py::ForwardMultiModel.__init__`,
which uses the **identical** formula, so this is not an approximation.

`sees=[source1]` / `sees=[source2]` encode "band 1 only shows source 1, band 2
only shows source 2" — the same per-band separation the old notebook's
`multiband_simulate` produced.


In [ ]:
dataset1 = ImageData(observed_image1, sim_config, background_rms=background_rms,
                    exp_time=exp_time, sees=[source1])
dataset2 = ImageData(observed_image2, sim_config, background_rms=background_rms,
                    exp_time=exp_time, sees=[source2])

prob_model = ProbModel(model, [dataset1, dataset2], mode="forward")
model_seq = ModellingSequence(prob_model)
ctx = InferenceContext.from_modelling_sequence(model_seq)

pipeline = Pipeline(ctx, seed=42)


## MAP

Old notebook: `optax.adabelief(1e-2, b1=0.95, b2=0.99)` (**no** `nesterov`),
`num_steps=4000`, `n_samples=1000`, `pbar_interval=100`, `seed=1`.

**Numerical-difference note:** the pipeline's built-in default MAP optimizer
(`gigalens_research.inference_utils.pipeline._default_map_optimizer`) adds
`nesterov=True` when the installed `optax` supports it — a silent behavior
change relative to the old notebook if you used the stage's default. This
cell passes an explicit `optimizer_factory` so the optimizer matches the old
notebook exactly (no Nesterov momentum).


In [ ]:
def _old_map_optimizer():
    return optax.adabelief(1e-2, b1=0.95, b2=0.99)


pipeline.add(MAPStage(
    num_steps=4000,
    n_samples=1000,
    pbar_interval=100,
    seed=1,
    # optimizer_factory=_old_map_optimizer,
    optimizer_id="adabelief_1e-2_b1_0.95_b2_0.99_no_nesterov",
))

def make_diag_qz(z_best):
    return tfd.MultivariateNormalDiag(
        loc=jnp.asarray(z_best),
        scale_diag=jnp.full(z_best.shape[-1], 1e-3),
    )


pipeline.add(BridgeStage(
    name="diag_qz_from_map",
    version="v1",
    requires=("z_best",),
    produces=("qz",),
    fn=make_diag_qz,
))

pipeline.add(MCLMCStage(
    n_chains=8,
    num_burnin_steps=20000,
    num_results=20000,
    desired_energy_variance=5e-4,
    seed=10,
    progress_bar=True,
    debug=True,
))


## Run

**Not executed in this notebook** (MAP + MCLMC is exactly the "heavy compute"
this port was told not to run on the login node). `Pipeline.run` also prints
and persists a **model card** (PSF / noise / grid / precision / trace mode) —
check it before trusting anything downstream, per this repo's operating card.


In [ ]:
results_dir = os.path.join(
    os.path.expanduser("~"), "GIGALens-Code", "results",
    "sample_cosmology", "dspl_cosmology_newapi",
)
artifacts = pipeline.run(out_dir=results_dir, resume=True)


In [ ]:
pipeline_report = PipelineReport(pipeline)
fig = pipeline_report.diagnostics("mclmc", chain=3)
fig.show()

report = PosteriorReport(pipeline.posterior(), truth_x=truth_scene)
# report.full_report()


In [ ]:
report.convergence_panel()
# report.source_comparison_panel()
report.z_score_panel(truth_x=truth_scene)
report.corner()#truth=truth_scene)

## Caveats and open flags (report, not fixes)

- **New-API validation status.** `docs/api-split.md` states the scene API
  "has not yet been validated" against the old API's real-system results.
  This notebook's construction (model/prior/prob-model wiring, the
  `deflection_ratio` formula, and the forward-noise formula) was checked
  line-by-line against the old package and smoke-tested end-to-end (model
  build → forward simulate → 2-step MAP → bridge `qz`), but a **numerical
  cross-check against the old notebook's own MAP/MCLMC output** has not been
  done (that requires the actual (unrun) compute). Treat agreement with the
  old notebook as unconfirmed until that comparison is run.
- **Likely degeneracy to watch (not diagnosed, not fixed):** in a
  double-source-plane system where the cosmological information comes
  entirely from the *relative* deflection-ratio scaling between two source
  planes sharing one lens, the mass-profile slope `gamma` trades off against
  `Om0`/`w0` in a way that's well documented in the double-source-plane
  cosmography literature (e.g. Collett & Auger 2014) — effectively a
  mass-sheet-transform-like degeneracy. If MCLMC mixes slowly here, a curved
  `gamma`–`Om0`–`w0` degeneracy is a reasonable first hypothesis to check
  (per this repo's own prior finding that curved degeneracies, not noise or
  multimodality, explain slow MCLMC mixing on the single-plane carousel
  system — see `docs/logs/carousel-mclmc-sampling.md`) — but this is an
  UNCERTIFIED hypothesis, not a finding; use `/diagnose-sampling` on the
  actual run before acting on it.
- **Two independent single-source bands, not a doubly-imaged system.** Each
  band shows only its own source (no shared/multiply-imaged background
  object), so the cosmological constraint comes purely from how much each
  band's single image is sheared/magnified by the shared lens at its own
  deflection ratio — not from comparing multiple images of the *same* source.
  This is how the old notebook built it too; noted here because it affects
  how tightly `Om0`/`w0` can plausibly be constrained (weaker than a true
  multiply-imaged double-source-plane lens).
